In [10]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
import numpy as np

In [ ]:
# 1. Load Dataset
# We'll use the 'billsum' dataset for summarization. It contains US Congressional and California state bills.
dataset_base = load_dataset("billsum", split="ca_test")

# Subsample for faster training (e.g., 200 examples)
dataset = dataset_base.select(range(200))

dataset = dataset.train_test_split(test_size=0.2)

# 2. Preprocess
checkpoint = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

prefix = "summarize: "

# 3. Load Model
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

# 4. Metrics
rouge = evaluate.load("rouge")

In [12]:

def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=216, truncation=True)

    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)

    return {k: round(v, 4) for k, v in result.items()}

# 5. Training Arguments
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

training_args = Seq2SeqTrainingArguments(
    output_dir="my_awesome_billsum_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4, # Reduced batch size for memory
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=False, # Set to True if using CUDA
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Evaluate before training
print("Metrics before training:")
print(trainer.evaluate())

# 6. Train
trainer.train()

# Evaluate after training
print("Metrics after training:")
print(trainer.evaluate())

Map: 100%|██████████| 20/20 [00:00<00:00, 84.62 examples/s] 

C:\Users\shahn\AppData\Local\Temp\ipykernel_6616\2628670496.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
C:\Users\shahn\AppData\Local\Temp\ipykernel_6616\2628670496.py:48: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Metrics before training:


{'eval_loss': 4.8525285720825195, 'eval_model_preparation_time': 0.008, 'eval_rouge1': 0.1545, 'eval_rouge2': 0.054, 'eval_rougeL': 0.1274, 'eval_rougeLsum': 0.128, 'eval_gen_len': 20.0, 'eval_runtime': 46.1733, 'eval_samples_per_second': 0.433, 'eval_steps_per_second': 0.108}


Epoch,Training Loss,Validation Loss,Model Preparation Time,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,No log,3.917365,0.008000,0.143700,0.050600,0.118600,0.118900,20.000000
2,No log,3.653472,0.008000,0.141500,0.048900,0.116500,0.116800,20.000000


d:\codes\summarizer\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Metrics after training:


{'eval_loss': 3.6534721851348877, 'eval_model_preparation_time': 0.008, 'eval_rouge1': 0.1415, 'eval_rouge2': 0.0489, 'eval_rougeL': 0.1165, 'eval_rougeLsum': 0.1168, 'eval_gen_len': 20.0, 'eval_runtime': 42.0652, 'eval_samples_per_second': 0.475, 'eval_steps_per_second': 0.119, 'epoch': 2.0}
